# Function 6: Cake and Stuff
Time to get cooking! You are optimising a cake recipe. There are five ingredients. The outputs correspond to the sum of different objectives: flavor, consistency, calories, waste and cost. Each objective receives negative points by our expert taster. You want this sum to be as close to zero as possible!

In [13]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
import matplotlib.pyplot as plt


In [14]:
def load_inputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    # Make the file a proper list of lists
    content = "[" + content.replace("]\n[", "],[") + "]"

    # Safe eval with restricted globals
    return eval(content, {"array": np.array})


def load_outputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    content = "[" + content.replace("]\n[", "],[") + "]"

    return eval(content, {"np": np})


def get_input_points(function_number, file_path="../inputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_inputs(file_path)
    index = function_number - 1

    inputs = [dataset[index] for dataset in data]
    return np.array(inputs)


def get_output_points(function_number, file_path="../outputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_outputs(file_path)
    index = function_number - 1

    outputs = [row[index] for row in data]
    return np.array(outputs)

# Load inputs
X = np.load(r'initial_inputs.npy')
y = np.load(r'initial_outputs.npy')


# Get input and outputs from submissions
inputs_array = get_input_points(6)
outputs_array = get_output_points(6)


# Append inputs_f1_array to X
X = np.vstack((X, inputs_array))

# Append outputs_f1_array to Y
y = np.hstack((y, outputs_array))
y = y.ravel()

print("New shape of X:", X.shape)
print("New shape of Y:", y.shape)

New shape of X: (31, 5)
New shape of Y: (31,)


In [15]:
# ----- Flatten outputs -----
y = y.ravel()

# ----- Fit GP -----
kernel = C(1.0, (1e-3, 1e5)) * Matern(
    length_scale=np.ones(X.shape[1]),
    length_scale_bounds=(1e-3, 2.0),
    nu=2.5,
) + WhiteKernel(noise_level=1e-8)
gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,
    normalize_y=True,
    random_state=42,
)
gp.fit(X, y)

rng = np.random.default_rng(42)

# ----- Current best recipe -----
best_idx = np.argmax(y)
best_point = X[best_idx]
print("Current best point:", best_point)
print("Current best output:", y[best_idx])

# ----- Local trust-region search around the top-performing region -----
top_indices = np.argsort(y)[-3:]
top_points = X[top_indices]
radius = 0.02
local_clouds = []
for point in top_points:
    local_clouds.append(point + rng.uniform(-radius, radius, size=(4000, X.shape[1])))

X_candidates = np.vstack(local_clouds)
X_candidates = np.clip(X_candidates, 0.01, 0.99)

# ----- Mild UCB score for mostly local exploitation -----
mean, std = gp.predict(X_candidates, return_std=True)
ucb = mean + 0.2 * std
next_query = X_candidates[np.argmax(ucb)]

# ----- Round and format next candidate -----
next_query = np.round(next_query, 6)
formatted_next_query = f"{next_query[0]:.6f}-{next_query[1]:.6f}-{next_query[2]:.6f}-{next_query[3]:.6f}-{next_query[4]:.6f}"
print("Next Query Point:", formatted_next_query)

Current best point: [0.884317 0.312451 0.668732 0.699154 0.156208]
Current best output: -0.5707448070355233
Next Query Point: 0.876866-0.329180-0.685775-0.716547-0.136781


/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
